# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [ ]:
# imports
import os
import requests
from openai import OpenAI
import ollama
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from dotenv import load_dotenv
import json

In [ ]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'
OLLAMA_API = "http://localhost:11434/chat"
HEADERS = {"ContentType" : "application/json"}

In [ ]:
# set up environment

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

In [ ]:
openai = OpenAI()
ollama_via_openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

In [ ]:
system_prompt="You are an online tutor. You need to idenfy the code snippet and explain how it function. Use full code examples in markdown"

In [ ]:
# here is the question; type over this to ask something new
def get_query_user_prompt(snippet, language):
    user_prompt = f"Here is the code snippet written in {language} - {snippet}. "
    user_prompt += "Please explain what this code does and why"
    return user_prompt

# yield from {book.get("author") for book in books if book.get("author")}

In [ ]:
def messages_for(snippet, language):
    return [
        { "role": "system", "content": system_prompt },
        { "role": "user", "content": get_query_user_prompt(snippet, language)}
    ]

In [ ]:
# Get gpt-4o-mini to answer, with streaming
def explain(snippet, language):
    stream = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=messages_for(snippet, language),
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)
    

In [ ]:
explain("yield from {book.get('author') for book in books if book.get('author')}", "Python")

In [ ]:
# Get Llama 3.2 to answer
def explain_llama(snippet, language):
    stream = ollama_via_openai.chat.completions.create(
        model=MODEL_LLAMA,
        messages=messages_for(snippet, language),
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
explain_llama("yield from {book.get('author') for book in books if book.get('author')}", "Python")

In [ ]:
explain_llama("yield* func1()", "Javascript")